In [1]:
# Cell 1 — Install dependencies
!pip install -q transformers accelerate bitsandbytes pillow fastapi uvicorn pyngrok nest-asyncio
!pip install -q "fschat[model_worker,webui]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.7/137.7 kB 7.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.9/256.9 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 805.0/805.0 kB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.9/73.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.1/67.1 kB 7.6 MB/s eta 0:00:00


In [2]:
# Cell 2 — Install Cloudflare tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print("Cloudflare tunnel ready ✅")

Cloudflare tunnel ready ✅


In [3]:
# Cell 3 — Load LLaVA model
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
from PIL import Image
import requests
from io import BytesIO

print("Loading LLaVA model... (this takes 3-5 minutes first time)")

model_id = "llava-hf/llava-1.5-7b-hf"

# 4-bit quantization so it fits in Colab T4 (15GB VRAM)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

from transformers import LlavaForConditionalGeneration, AutoProcessor

processor = AutoProcessor.from_pretrained(model_id)
model = LlavaForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)
model.eval()
print("LLaVA loaded ✅")

Loading LLaVA model... (this takes 3-5 minutes first time)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/701 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/674 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/686 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

LLaVA loaded ✅


In [ ]:
# Cell 4 — FastAPI server + Cloudflare tunnel (Colab fixed)
import subprocess
import threading
import time
import base64
import re
import nest_asyncio
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from PIL import Image
import io
import torch
import json

nest_asyncio.apply()

app = FastAPI()

PHISHING_PROMPT = """You are a cybersecurity expert analyzing website screenshots for phishing.
Look for these phishing indicators:
- Fake login forms asking for credentials
- Urgency messages (account suspended, verify now)
- Mismatched branding or logos
- Suspicious password/credit card input fields
- Poor grammar or spelling
- Fake security badges

Analyze this screenshot and respond with ONLY a JSON object:
{
  "image_score": <float 0.0 to 1.0>,
  "reason": "<brief explanation>",
  "indicators": ["<indicator1>", "<indicator2>"]
}
Where 1.0 = definitely phishing, 0.0 = definitely safe."""

class ScreenshotRequest(BaseModel):
    screenshot_base64: str
    url: str = ""

@app.get("/")
def root():
    return {"status": "LLaVA service running"}

@app.post("/analyze")
async def analyze(request: ScreenshotRequest):
    try:
        image_data = base64.b64decode(request.screenshot_base64)
        image = Image.open(io.BytesIO(image_data)).convert("RGB")

        conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": PHISHING_PROMPT}
                ]
            }
        ]

        prompt = processor.apply_chat_template(
            conversation,
            add_generation_prompt=True
        )
        inputs = processor(
            images=image,
            text=prompt,
            return_tensors="pt"
        ).to(model.device, torch.float16)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=False,
                temperature=1.0
            )

        response_text = processor.decode(
            output[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        )

        json_match = re.search(r'\{.*\}', response_text, re.DOTALL)
        if json_match:
            result = json.loads(json_match.group())
            return {
                "image_score": float(result.get("image_score", 0.5)),
                "llava_response": result.get("reason", ""),
                "indicators": result.get("indicators", [])
            }
        else:
            return {
                "image_score": 0.5,
                "llava_response": response_text,
                "indicators": []
            }

    except Exception as e:
        return {"image_score": 0.5, "llava_response": f"Error: {str(e)}", "indicators": []}



def start_tunnel():
    time.sleep(4)
    result = subprocess.Popen(
        ["./cloudflared", "tunnel", "--url", "http://localhost:8100"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    for line in result.stdout:
        line = line.decode()
        if "trycloudflare.com" in line:
            url = re.search(r'https://[^\s]+trycloudflare\.com', line)
            if url:
                tunnel_url = url.group()
                print(f"\n{'='*50}")
                print(f"✅ TUNNEL URL: {tunnel_url}")
                print(f"{'='*50}")
                print(f"LLAVA_API_URL={tunnel_url}")
                with open("tunnel_url.txt", "w") as f:
                    f.write(tunnel_url)

tunnel_thread = threading.Thread(target=start_tunnel, daemon=True)
tunnel_thread.start()

config = uvicorn.Config(app, host="0.0.0.0", port=8100, loop="asyncio")
server = uvicorn.Server(config)

print("Starting LLaVA API server...")
await server.serve()


INFO:     Started server process [21577]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8100 (Press CTRL+C to quit)


Starting LLaVA API server...

✅ TUNNEL URL: https://talented-mary-miss-practitioners.trycloudflare.com
LLAVA_API_URL=https://talented-mary-miss-practitioners.trycloudflare.com
INFO:     103.199.188.243:0 - "POST /analyze HTTP/1.1" 200 OK
